# Taxonomy classification

Reflectance relative to the Sun (normalised to $g$) and spectral gradients from the absolute magnitudes $H_f$.

In [2]:
import itertools
import numpy as np
import pandas as pd

df = pd.read_csv('../medellin-vroc-latest.csv')
df.shape

(524, 120)

In [5]:
df[df['mpc_designation'] == '2026 D014']

,provid,permid,mpc_designation,epoch_utc_ini,epoch_utc_end,n_obs,n_obs_u,n_obs_g,n_obs_r,n_obs_i,...,spec_T_sbdb,spec_T_sbdb_err,rot_per_lsm_gst,rot_per_fr_gst,res_body,res_flag,res_type,res_kp,res_k,res_orbit


## 1. Reflectance relative to the Sun

$$R_{f,g} = 10^{-0.4\left[(M_f - M_g) - (M_f^{\odot} - M_g^{\odot})\right]}$$

$M_f$ are the `H_*` columns. Only $g, r, i, z$ have solar magnitudes, so $u$ and $y$ are not computed. By construction $R_{g,g} = 1$.

In [2]:
# Solar absolute magnitudes per band
M_sun = {'g': -26.34, 'r': -27.04, 'i': -27.38, 'z': -27.56}

# Effective wavelengths (nm)
lam = {'u': 372.4, 'g': 480.7, 'r': 622.1, 'i': 755.9, 'z': 868.0, 'y': 975.3}

bands = list(M_sun)  # g, r, i, z

for f in bands:
    color = df[f'H_{f}'] - df['H_g']
    color_sun = M_sun[f] - M_sun['g']
    df[f'R_{f}g'] = 10 ** (-0.4 * (color - color_sun))

df[['provid'] + [f'R_{f}g' for f in bands]].describe()

,R_gg,R_rg,R_ig,R_zg
count,513.0,513.000000,513.000000,94.000000
mean,1.0,0.871000,0.743026,0.648760
std,0.0,0.134660,0.119496,0.140107
min,1.0,0.496405,0.440249,0.474227
25%,1.0,0.784222,0.661459,0.568583
50%,1.0,0.859393,0.740848,0.620226
75%,1.0,0.939486,0.819785,0.680513
max,1.0,1.686283,1.201592,1.230155


## 2. Spectral gradient

$$S(\lambda_1, \lambda_2) = 10^4 \, \frac{R_{f_2,g} - R_{f_1,g}}{|\lambda_2 - \lambda_1|}$$

Computed for every pair of bands with $\lambda_1 < \lambda_2$. Columns are named `S_{f1}{f2}`.

In [3]:
for f1, f2 in itertools.combinations(bands, 2):
    df[f'S_{f1}{f2}'] = 1e4 * (df[f'R_{f2}g'] - df[f'R_{f1}g']) / abs(lam[f2] - lam[f1])

S_cols = [c for c in df.columns if c.startswith('S_')]
df[S_cols].describe()

,S_gr,S_gi,S_gz,S_ri,S_rz,S_iz
count,513.000000,513.000000,94.000000,513.000000,94.000000,94.000000
mean,-9.123075,-9.337716,-9.068939,-9.564549,-8.551262,-6.604488
std,9.523350,4.342162,3.617537,8.667251,5.518221,11.562090
min,-35.614956,-20.339798,-13.575346,-70.520870,-24.720170,-27.750597
25%,-15.260092,-12.301618,-11.139082,-13.207326,-10.588063,-12.428694
50%,-9.943939,-9.416846,-9.805674,-9.074677,-8.662079,-7.978037
75%,-4.279627,-6.548521,-8.249082,-4.906287,-6.069969,-3.698879
max,48.534875,7.325299,5.942559,16.547331,18.050864,51.680403


In [4]:
R_cols = [f'R_{f}g' for f in bands]
df[['provid'] + R_cols + S_cols].head(10)

,provid,R_gg,R_rg,R_ig,R_zg,S_gr,S_gi,S_gz,S_ri,S_rz,S_iz
0,1978 UU1,1.0,0.891232,0.828541,NaN,-7.692255,-6.230340,NaN,-4.685386,NaN,NaN
1,1981 EY23,1.0,0.667749,0.628955,NaN,-23.497269,-13.482740,NaN,-2.899374,NaN,NaN
2,1986 PX5,1.0,1.283513,1.087103,NaN,20.050457,3.165068,NaN,-14.679431,NaN,NaN
3,1992 EP8,1.0,0.915921,0.822018,NaN,-5.946215,-6.467360,NaN,-7.018106,NaN,NaN
4,1997 JU3,1.0,1.021786,0.821870,NaN,1.540760,-6.472759,NaN,-14.941455,NaN,NaN
5,1998 BU48,1.0,0.804337,1.015652,NaN,-13.837564,0.568762,NaN,15.793384,NaN,NaN
6,1998 FP73,1.0,0.799311,0.617932,NaN,-14.193031,-13.883293,NaN,-13.555962,NaN,NaN
7,1998 UP5,1.0,0.862752,0.710493,NaN,-9.706362,-10.519869,NaN,-11.379584,NaN,NaN
8,1999 JO129,1.0,0.872588,0.552664,0.508299,-9.010726,-16.254941,-12.695601,-23.910636,-14.814517,-3.957598
9,1999 RM160,1.0,0.729851,0.586809,0.495878,-19.105291,-15.014193,-13.016305,-10.690715,-9.514952,-8.111589
